In [17]:
# ============================================================
# CELL 0 — COLAB SETUP
# Run this FIRST at the start of every new Colab session.
# Takes ~1 min. Safe to re-run.
# ============================================================

# 1. Mount Google Drive
import os
from dotenv import load_dotenv
load_dotenv()
import shutil
from google.colab import drive

mountpoint = '/content/drive'

# Unmount and clean up any stale mount
try:
    drive.flush_and_unmount()
except Exception:
    pass  # Ignore if not mounted yet

# Remove stale directory so mount() gets a clean slate
if os.path.exists(mountpoint):
    shutil.rmtree(mountpoint)

drive.mount(mountpoint)

# 2. Install packages not pre-installed on Colab
!pip install praat-parselmouth noisereduce soundfile transformers torchaudio -q

# 3. Create all project directories on Drive (idempotent)
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

dirs = [
    "outputs",
    "data/train/native",
    "data/train/non_native",
    "data/preprocessed/native",
    "data/preprocessed/non_native",
    "data/embeddings",
    "data/segments/native",
    "data/segments/non_native",
    "data/augmented/native",
    "data/augmented/non_native",
]
for d in dirs:
    os.makedirs(f"{PROJECT_ROOT}/{d}", exist_ok=True)

# 4. Verify GPU (critical for Stage 6A)
import torch
print(f"\nGPU available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU — Runtime → Change runtime type → T4 GPU")
    print("   Stage 6A will be very slow on CPU.")

# 5. Check renan_dataset.csv is on Drive
csv_path = f"{PROJECT_ROOT}/renan_dataset.csv"
if os.path.exists(csv_path):
    import pandas as pd
    df = pd.read_csv(csv_path)
    print(f"\n✅ renan_dataset.csv found ({len(df)} rows)")
else:
    print(f"\n❌ renan_dataset.csv NOT found.")
    print(f"   Upload it manually to: {PROJECT_ROOT}/")
    print(f"   (Google Drive → team_databaes folder → Upload file)")

print(f"\nProject root  : {PROJECT_ROOT}")
print(f"Setup complete ✅")

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 105.2 MB/s eta 0:00:00

GPU available : True
GPU           : Tesla T4
VRAM          : 15.6 GB

✅ renan_dataset.csv found (160 rows)

Project root  : /content/drive/MyDrive/team_databaes
Setup complete ✅


In [18]:
# ============================================================
# CELL 1 — STAGE 1: DATA EXPLORATION
# Reads renan_dataset.csv from Drive.
# All outputs saved directly to Drive.
# Safe to re-run — overwrites previous outputs.
# ============================================================

import os, io, time, requests, librosa, numpy as np, pandas as pd
from tqdm.notebook import tqdm

PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"
EXCEL_PATH   = f"{PROJECT_ROOT}/renan_dataset.csv"
OUTPUT_DIR   = f"{PROJECT_ROOT}/outputs"
TARGET_SR    = 16000

# Set to None to explore all 160 files (takes ~5 min)
# Keep at 50 for a quick sanity check (~1 min)
EXPLORE_N = None

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Load spreadsheet ──────────────────────────────────────────
print("📂 Loading dataset spreadsheet...")
df_raw = pd.read_csv(EXCEL_PATH)
print(f"   Total rows loaded: {len(df_raw)}")
print(f"   Columns: {list(df_raw.columns)}")

df_raw["label"] = df_raw["nativity_status"].map({"Native": 1, "Non-Native": 0})

unknown_labels = df_raw[df_raw["label"].isna()]
if len(unknown_labels) > 0:
    print(f"⚠️  {len(unknown_labels)} rows with unrecognised nativity_status:")
    print(unknown_labels["nativity_status"].unique())

df_explore = df_raw.sample(n=EXPLORE_N, random_state=42) if EXPLORE_N else df_raw
print(f"\n🔍 Exploring {len(df_explore)} files...\n")

# ── Download + inspect ────────────────────────────────────────
def download_and_inspect(row):
    dp_id   = row["dp_id"]
    url     = row["audio_url"]
    label   = row["label"]
    dialect = row["language"]

    result = {
        "dp_id":    dp_id, "label": label,
        "nativity": row["nativity_status"], "dialect": dialect, "url": url,
    }
    try:
        response    = requests.get(url, timeout=15)
        response.raise_for_status()
        audio_bytes = io.BytesIO(response.content)

        y, sr = librosa.load(audio_bytes, sr=None, mono=True)
        duration      = librosa.get_duration(y=y, sr=sr)
        rms           = float(np.sqrt(np.mean(y ** 2)))
        zcr           = float(np.mean(librosa.feature.zero_crossing_rate(y)))
        frame_rms     = librosa.feature.rms(y=y)[0]
        silence_ratio = float(np.mean(frame_rms < 0.01 * frame_rms.max()))

        result.update({
            "original_sr":    sr,
            "duration_s":     round(duration, 2),
            "rms_volume":     round(rms, 5),
            "zero_cross_rate":round(zcr, 5),
            "silence_ratio":  round(silence_ratio, 3),
            "n_samples":      len(y),
            "usable":         duration >= 3.0,
            "error":          None,
        })
    except requests.exceptions.Timeout:
        result.update({"error": "TIMEOUT", "usable": False})
    except requests.exceptions.HTTPError as e:
        result.update({"error": f"HTTP_{e.response.status_code}", "usable": False})
    except Exception as e:
        result.update({"error": str(e)[:80], "usable": False})
    return result

records = []
for _, row in tqdm(df_explore.iterrows(), total=len(df_explore), desc="Inspecting audio"):
    records.append(download_and_inspect(row))
    time.sleep(0.05)

df_stats = pd.DataFrame(records)

# ── Print report ──────────────────────────────────────────────
sep = "=" * 58
report_lines = []

def log(line=""):
    print(line)
    report_lines.append(line)

log(sep)
log("  STAGE 1 — DATA EXPLORATION REPORT")
log(f"  Dataset: renan_dataset.csv  |  Files inspected: {len(df_stats)}")
log(sep)

log("\n📊 CLASS BALANCE")
log("-" * 35)
balance = df_raw["nativity_status"].value_counts()
for name, count in balance.items():
    pct = 100 * count / len(df_raw)
    log(f"   {name:<15}: {count:>5} files  ({pct:.1f}%)")
imbalance_ratio = balance.max() / balance.min()
log(f"\n   Imbalance ratio: {imbalance_ratio:.1f}x")
if imbalance_ratio > 2:
    log("   ⚠️  Significant imbalance — class_weight='balanced' is essential in Stage 8")

log("\n🌍 DIALECT DISTRIBUTION")
log("-" * 35)
for dialect, count in df_raw["language"].value_counts().items():
    pct = 100 * count / len(df_raw)
    log(f"   {dialect:<18}: {count:>4} files  ({pct:.1f}%)")

log("\n⏱️  DURATION STATS (seconds)")
log("-" * 35)
valid = df_stats[df_stats["error"].isna()]
for label_val, label_name in [(1, "Native"), (0, "Non-Native")]:
    subset = valid[valid["label"] == label_val]["duration_s"]
    if len(subset) == 0:
        continue
    log(f"\n   {label_name}:")
    log(f"     mean={subset.mean():.1f}s  median={subset.median():.1f}s  "
        f"min={subset.min():.1f}s  max={subset.max():.1f}s")
    est_segs = subset.apply(lambda d: max(0, int((d - 3) / 1) + 1))
    log(f"     Estimated 3s segments per file: ~{est_segs.mean():.0f} avg, "
        f"{est_segs.sum()} total")

log("\n🎵 SAMPLE RATES FOUND (original, before resampling)")
log("-" * 35)
for sr, count in valid["original_sr"].value_counts().items():
    flag = "✅" if sr == TARGET_SR else "⚠️  will resample"
    log(f"   {sr} Hz : {count} files  {flag}")

log("\n🔊 VOLUME (RMS) BY CLASS")
log("-" * 35)
rms_stats = valid.groupby("label")["rms_volume"].describe()
log(rms_stats.round(5).to_string())
rms_native    = valid[valid["label"]==1]["rms_volume"].mean()
rms_nonnative = valid[valid["label"]==0]["rms_volume"].mean()
if abs(rms_native - rms_nonnative) / max(rms_native, rms_nonnative) > 0.2:
    log("   ⚠️  Noticeable volume difference — RMS normalization in Stage 3 is important")

log("\n🤫 SILENCE RATIO BY CLASS  (0=no silence, 1=all silence)")
log("-" * 35)
log(valid.groupby("label")["silence_ratio"].describe().round(3).to_string())

log("\n⚠️  PROBLEMATIC FILES")
log("-" * 35)
errors = df_stats[df_stats["error"].notna()]
short  = df_stats[(df_stats["error"].isna()) & (df_stats["duration_s"] < 3.0)]
log(f"   Download/load errors : {len(errors)}")
log(f"   Too short (<3s)      : {len(short)}  ← will be dropped in Stage 3")
log(f"   Usable files         : {df_stats['usable'].sum()} / {len(df_stats)}")
if len(errors) > 0:
    log("\n   Error breakdown:")
    log(errors["error"].value_counts().to_string())

log("\n" + sep)
log("  KEY TAKEAWAYS FOR DOWNSTREAM STAGES")
log(sep)
log(f"  Stage 3: Resample all audio to {TARGET_SR}Hz mono + RMS normalize")
log(f"  Stage 4: Use 3s window / 1s hop for segmentation")
log(f"  Stage 5: Augment training data only (non-native is minority class)")
log(f"  Stage 8: Use class_weight='balanced' in SVM due to imbalance")
log(sep)

# ── Save outputs directly to Drive ───────────────────────────
df_stats.to_csv(f"{OUTPUT_DIR}/stage1_exploration.csv", index=False)
log(f"\n💾 Saved: {OUTPUT_DIR}/stage1_exploration.csv")

with open(f"{OUTPUT_DIR}/stage1_report.txt", "w") as f:
    f.write("\n".join(report_lines))
log(f"💾 Saved: {OUTPUT_DIR}/stage1_report.txt")

print("\n✅ Stage 1 complete.")

📂 Loading dataset spreadsheet...
   Total rows loaded: 160
   Columns: ['dp_id', 'audio_url', 'nativity_status', 'language']

🔍 Exploring 160 files...



Inspecting audio:   0%|          | 0/160 [00:00<?, ?it/s]

  STAGE 1 — DATA EXPLORATION REPORT
  Dataset: renan_dataset.csv  |  Files inspected: 160

📊 CLASS BALANCE
-----------------------------------
   Native         :   114 files  (71.2%)
   Non-Native     :    46 files  (28.8%)

   Imbalance ratio: 2.5x
   ⚠️  Significant imbalance — class_weight='balanced' is essential in Stage 8

🌍 DIALECT DISTRIBUTION
-----------------------------------
   Arabic_SA         :   63 files  (39.4%)
   Arabic_QA         :   56 files  (35.0%)
   Arabic_AE         :   38 files  (23.8%)
   Arabic_KW         :    3 files  (1.9%)

⏱️  DURATION STATS (seconds)
-----------------------------------

   Native:
     mean=45.4s  median=42.1s  min=17.2s  max=118.3s
     Estimated 3s segments per file: ~43 avg, 4640 total

   Non-Native:
     mean=54.0s  median=36.7s  min=25.6s  max=400.2s
     Estimated 3s segments per file: ~51 avg, 2214 total

🎵 SAMPLE RATES FOUND (original, before resampling)
-----------------------------------
   16000.0 Hz : 107 files  ✅
   48000

In [19]:
# ============================================================
# CELL 2 — STAGE 2: DATA COLLECTION
# Part A: Download all 160 Renan MP3s → Drive
# Part B: Fetch 200 CV native clips via Mozilla API (streaming)
#         ✅ NO full 3.25GB download — streams tar and extracts
#            only the ~200 clips we need (~10MB total written)
# Part C: Build master manifest → Drive
#
# ── BEFORE RUNNING ──────────────────────────────────────────
# 1. Get your API key from:
#    https://datacollective.mozillafoundation.org
#    (Account → API Keys → Create)
# 2. Paste it into CV_API_KEY below
# ────────────────────────────────────────────────────────────
# Safe to re-run — skips already-downloaded files.
# ============================================================

import os, io, time, tarfile, hashlib, requests
import numpy as np, pandas as pd
from tqdm.notebook import tqdm

PROJECT_ROOT    = "/content/drive/MyDrive/team_databaes"
RENAN_CSV       = f"{PROJECT_ROOT}/renan_dataset.csv"
TRAIN_NATIVE    = f"{PROJECT_ROOT}/data/train/native"
TRAIN_NONNATIVE = f"{PROJECT_ROOT}/data/train/non_native"
OUTPUT_DIR      = f"{PROJECT_ROOT}/outputs"

# ── ⚠️  PASTE YOUR API KEY HERE ──────────────────────────────
CV_API_KEY = os.getenv("CV_API_KEY", "YOUR_API_KEY_HERE")
# ─────────────────────────────────────────────────────────────

CV_DATASET_ID  = "cmj8u3os6000tnxxb169x1zdc"
CV_API_URL     = f"https://datacollective.mozillafoundation.org/api/datasets/{CV_DATASET_ID}/download"
CV_NATIVE_N    = 200
CV_MIN_UPVOTES = 2
TIMEOUT        = 20
SLEEP          = 0.1

# ════════════════════════════════════════════════════════════
# PART A — DOWNLOAD RENAN FILES
# ════════════════════════════════════════════════════════════
print("=" * 58)
print("  PART A — Downloading Renan Training Audio (160 files)")
print("=" * 58)

df_renan = pd.read_csv(RENAN_CSV)
df_renan["label"]     = df_renan["nativity_status"].map({"Native": 1, "Non-Native": 0})
df_renan["filename"]  = df_renan["dp_id"].apply(lambda x: f"dp_{x}.mp3")
df_renan["save_dir"]  = df_renan["label"].map({1: TRAIN_NATIVE, 0: TRAIN_NONNATIVE})
df_renan["save_path"] = df_renan.apply(
    lambda r: os.path.join(r["save_dir"], r["filename"]), axis=1
)

manifest_records = []

def download_renan_file(url, save_path, dp_id):
    if os.path.exists(save_path):
        return {
            "dp_id": dp_id, "status": "already_exists",
            "file_size_kb": round(os.path.getsize(save_path) / 1024, 1),
            "md5": None, "error": None
        }
    try:
        r = requests.get(url, timeout=TIMEOUT)
        r.raise_for_status()
        ctype = r.headers.get("Content-Type", "")
        if "audio" not in ctype and "octet-stream" not in ctype:
            return {
                "dp_id": dp_id, "status": "bad_content_type",
                "file_size_kb": 0, "md5": None, "error": f"Content-Type={ctype}"
            }
        with open(save_path, "wb") as f:
            f.write(r.content)
        return {
            "dp_id": dp_id, "status": "downloaded",
            "file_size_kb": round(len(r.content) / 1024, 1),
            "md5": hashlib.md5(r.content).hexdigest()[:8], "error": None
        }
    except requests.exceptions.Timeout:
        return {"dp_id": dp_id, "status": "error", "file_size_kb": 0, "md5": None, "error": "TIMEOUT"}
    except requests.exceptions.HTTPError as e:
        return {"dp_id": dp_id, "status": "error", "file_size_kb": 0, "md5": None,
                "error": f"HTTP_{e.response.status_code}"}
    except Exception as e:
        return {"dp_id": dp_id, "status": "error", "file_size_kb": 0, "md5": None,
                "error": str(e)[:80]}

for _, row in tqdm(df_renan.iterrows(), total=len(df_renan), desc="Renan files"):
    result = download_renan_file(row["audio_url"], row["save_path"], row["dp_id"])
    result.update({
        "source":    "renan",
        "label":     row["label"],
        "nativity":  row["nativity_status"],
        "dialect":   row["language"],
        "save_path": row["save_path"],
    })
    manifest_records.append(result)
    time.sleep(SLEEP)

a_df = pd.DataFrame(manifest_records)
print(f"\nRenan download complete:")
print(f"   Downloaded     : {(a_df['status']=='downloaded').sum()}")
print(f"   Already existed: {(a_df['status']=='already_exists').sum()}")
print(f"   Failed         : {(a_df['status'].isin(['error','bad_content_type'])).sum()}")


# ════════════════════════════════════════════════════════════
# PART B — MOZILLA COMMON VOICE 24.0 via API (streaming)
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 58)
print("  PART B — Common Voice 24.0 Arabic via API (streaming)")
print("=" * 58)

cv_records = []

if CV_API_KEY == "YOUR_API_KEY_HERE":
    print("⚠️  CV_API_KEY not set — skipping Part B.")
    print("   Set your API key at the top of this cell and re-run.")
    print("   Continuing with Renan-only data for now.")
else:
    # ── Step 1: Get presigned download URL from API ───────────
    print("Requesting presigned download URL from Mozilla API...")
    try:
        resp = requests.post(
            CV_API_URL,
            headers={
                "Authorization": f"Bearer {CV_API_KEY}",
                "Content-Type":  "application/json",
            },
            timeout=30,
        )
        resp.raise_for_status()
        download_url = resp.json()["downloadUrl"]
        print(f"✅ Got presigned URL (valid for ~15 min)")
    except requests.exceptions.HTTPError as e:
        print(f"❌ API error: HTTP {e.response.status_code}")
        print(f"   Response: {e.response.text[:300]}")
        print("   Check your API key and try again.")
        download_url = None
    except KeyError:
        print(f"❌ Unexpected API response — 'downloadUrl' not in response.")
        print(f"   Full response: {resp.json()}")
        download_url = None
    except Exception as e:
        print(f"❌ Request failed: {e}")
        download_url = None

    if download_url:
        # ── Step 2: PASS 1 — Stream tar to get train.tsv only ──
        # We stream the entire tar.gz but only materialise the TSV.
        # This reads sequentially until we hit the TSV member then stops.
        # Memory usage: only the TSV content (~few MB).
        print("\nPass 1: Streaming archive to extract train.tsv...")
        print("(Reads sequentially until TSV is found — stops early)")

        class StreamingTarWrapper(io.RawIOBase):
            """
            Wraps a requests streaming response so tarfile can read it
            as a file-like object without buffering the entire download.
            """
            def __init__(self, response):
                self._iter    = response.iter_content(chunk_size=1024 * 256)  # 256KB chunks
                self._buf     = b""
                self._pos     = 0
                self.bytes_read = 0

            def readable(self):
                return True

            def readinto(self, b):
                while len(self._buf) < len(b):
                    try:
                        chunk = next(self._iter)
                        self._buf += chunk
                        self.bytes_read += len(chunk)
                    except StopIteration:
                        break
                n = min(len(b), len(self._buf))
                b[:n] = self._buf[:n]
                self._buf = self._buf[n:]
                return n

        # --- Pass 1: extract train.tsv ---
        df_filtered  = None
        clips_prefix = None

        stream_resp1 = requests.get(download_url, stream=True, timeout=120)
        stream_resp1.raise_for_status()
        wrapper1 = StreamingTarWrapper(stream_resp1)

        with tarfile.open(fileobj=wrapper1, mode="r|gz") as tar:
            for member in tar:
                # Grab clips prefix ONLY from Arabic clips (/ar/clips/)
                # Do NOT use first .mp3 found — that is likely a non-Arabic locale
                if (clips_prefix is None and
                        member.name.endswith(".mp3") and
                        "/ar/clips/" in member.name):
                    clips_prefix = member.name.rsplit("/ar/clips/", 1)[0] + "/ar/clips/"
                    print(f"  Detected clips prefix: {clips_prefix}")

                if member.name.endswith("ar/train.tsv"):
                    print(f"  Found TSV: {member.name} ({member.size / 1e6:.1f} MB)")
                    f_obj = tar.extractfile(member)
                    # pd.read_csv needs a seekable fileobj — streaming tar members
                    # are not seekable, so we read the bytes fully into a BytesIO
                    # buffer first. TSV is ~9MB so this is fine in memory.
                    tsv_bytes = io.BytesIO(f_obj.read())
                    df_cv = pd.read_csv(tsv_bytes, sep="\t")
                    print(f"  Train clips in TSV: {len(df_cv):,}")
                    print(f"  Columns: {list(df_cv.columns)}")

                    # Quality filter
                    df_cv["up_votes"]   = pd.to_numeric(df_cv["up_votes"],   errors="coerce").fillna(0)
                    df_cv["down_votes"] = pd.to_numeric(df_cv["down_votes"], errors="coerce").fillna(0)
                    df_filtered = df_cv[
                        (df_cv["up_votes"]   >= CV_MIN_UPVOTES) &
                        (df_cv["down_votes"] == 0)
                    ].copy()
                    print(f"  After quality filter: {len(df_filtered):,}")

                    # CV 24.0 uses 'accents' (plural) not 'accent' (singular)
                    # Detect whichever is present
                    if "accents" in df_filtered.columns:
                        accent_col = "accents"
                    elif "accent" in df_filtered.columns:
                        accent_col = "accent"
                    else:
                        # No accent column at all — treat everything as unspecified
                        accent_col = None
                        df_filtered["_accent"] = "unspecified"
                        accent_col = "_accent"

                    print(f"  Using accent column: '{accent_col}'")

                    # Accent-stratified sampling
                    df_filtered["accent"] = (
                        df_filtered[accent_col].fillna("unspecified").replace("", "unspecified")
                    )
                    accent_groups = df_filtered["accent"].unique()
                    n_groups      = len(accent_groups)
                    per_group     = max(1, CV_NATIVE_N // n_groups)
                    remainder     = CV_NATIVE_N - (per_group * n_groups)

                    sampled_parts = []
                    for i, accent in enumerate(accent_groups):
                        grp    = df_filtered[df_filtered["accent"] == accent]
                        n_take = min(per_group + (1 if i < remainder else 0), len(grp))
                        sampled_parts.append(grp.sample(n=n_take, random_state=42))

                    df_sample = pd.concat(sampled_parts, ignore_index=True)
                    print(f"\n  Accent breakdown:")
                    print(df_filtered["accent"].value_counts().to_string())
                    print(f"\n  Sampled {len(df_sample)} clips across {n_groups} accent groups")
                    break   # ← stop streaming as soon as we have the TSV

        stream_resp1.close()
        print(f"  Pass 1 complete — read {wrapper1.bytes_read / 1e6:.0f} MB of archive")

        if df_filtered is None:
            print("❌ Could not find ar/train.tsv in archive. Skipping CV.")
        else:
            # Build set of clip paths we need
            accent_to_dialect = {
                "saudi":       "Arabic_SA",
                "gulf":        "Arabic_QA",
                "egyptian":    "Arabic_EG",
                "levantine":   "Arabic_LEV",
                "moroccan":    "Arabic_MA",
                "unspecified": "Arabic_MSA",
            }

            # Build a lookup: tar path → (out_path, dialect, accent, idx, up_votes)
            clips_needed = {}
            for idx, row in df_sample.iterrows():
                accent_slug      = str(row["accent"]).replace(" ", "_").lower()
                out_fname        = f"cv_{accent_slug}_{row['path']}"
                out_path         = os.path.join(TRAIN_NATIVE, out_fname)
                clip_path_in_tar = (clips_prefix or "cv-corpus-24.0-2025-12-05/ar/clips/") + row["path"]
                dialect_label    = accent_to_dialect.get(str(row["accent"]).lower(), "Arabic_MSA")

                clips_needed[clip_path_in_tar] = {
                    "dp_id":     f"cv_{idx:06d}",
                    "out_path":  out_path,
                    "dialect":   dialect_label,
                    "cv_accent": row["accent"],
                    "up_votes":  int(row["up_votes"]),
                }

            # Debug: print a sample of the expected tar paths so we can verify
            sample_keys = list(clips_needed.keys())[:3]
            print(f"  Sample expected tar paths:")
            for k in sample_keys:
                print(f"    {k}")

            # Skip clips already on Drive
            already_on_disk = {
                k: v for k, v in clips_needed.items()
                if os.path.exists(v["out_path"])
            }
            to_fetch = {
                k: v for k, v in clips_needed.items()
                if not os.path.exists(v["out_path"])
            }

            print(f"\nClips already on Drive : {len(already_on_disk)}")
            print(f"Clips to fetch         : {len(to_fetch)}")

            # Add already-on-disk clips to cv_records immediately
            for tar_path, info in already_on_disk.items():
                cv_records.append({
                    "dp_id":     info["dp_id"],
                    "source":    "mozilla_common_voice_24",
                    "label":     1,
                    "nativity":  "Native",
                    "dialect":   info["dialect"],
                    "save_path": info["out_path"],
                    "status":    "already_exists",
                    "error":     None,
                    "cv_accent": info["cv_accent"],
                    "up_votes":  info["up_votes"],
                })

            if to_fetch:
                # ── Step 3: PASS 2 — Stream again, save only needed clips ──
                # We stream the full archive but only write out the ~200
                # target clips. Everything else is discarded in memory.
                # Total written to disk: ~200 clips × ~50KB = ~10MB.
                print(f"\nPass 2: Streaming archive to extract {len(to_fetch)} clips...")
                print("(Reads full archive sequentially — ~3.25GB streamed, ~10MB written)")

                # Need a fresh presigned URL if the old one might have expired
                # (Pass 1 may have taken >15 min for large archives)
                print("Refreshing presigned URL for Pass 2...")
                try:
                    resp2 = requests.post(
                        CV_API_URL,
                        headers={
                            "Authorization": f"Bearer {CV_API_KEY}",
                            "Content-Type":  "application/json",
                        },
                        timeout=30,
                    )
                    resp2.raise_for_status()
                    download_url2 = resp2.json()["downloadUrl"]
                    print("✅ Got fresh presigned URL")
                except Exception as e:
                    print(f"⚠️  Could not refresh URL: {e} — trying original URL")
                    download_url2 = download_url

                stream_resp2 = requests.get(download_url2, stream=True, timeout=120)
                stream_resp2.raise_for_status()
                wrapper2 = StreamingTarWrapper(stream_resp2)

                extracted = 0
                errors    = 0
                pbar      = tqdm(total=len(to_fetch), desc="Extracting CV clips")

                with tarfile.open(fileobj=wrapper2, mode="r|gz") as tar:
                    for member in tar:
                        if member.name not in to_fetch:
                            # Skip — but we must still read past it in the stream
                            # tarfile does this automatically for r| (streaming) mode
                            continue

                        info  = to_fetch[member.name]
                        f_obj = tar.extractfile(member)

                        if f_obj is None:
                            cv_records.append({
                                "dp_id":     info["dp_id"],
                                "source":    "mozilla_common_voice_24",
                                "label":     1,
                                "nativity":  "Native",
                                "dialect":   info["dialect"],
                                "save_path": None,
                                "status":    "error",
                                "error":     "Empty tar member",
                                "cv_accent": info["cv_accent"],
                                "up_votes":  info["up_votes"],
                            })
                            errors += 1
                        else:
                            try:
                                with open(info["out_path"], "wb") as out_f:
                                    out_f.write(f_obj.read())
                                cv_records.append({
                                    "dp_id":     info["dp_id"],
                                    "source":    "mozilla_common_voice_24",
                                    "label":     1,
                                    "nativity":  "Native",
                                    "dialect":   info["dialect"],
                                    "save_path": info["out_path"],
                                    "status":    "extracted",
                                    "error":     None,
                                    "cv_accent": info["cv_accent"],
                                    "up_votes":  info["up_votes"],
                                })
                                extracted += 1
                            except Exception as e:
                                cv_records.append({
                                    "dp_id":     info["dp_id"],
                                    "source":    "mozilla_common_voice_24",
                                    "label":     1,
                                    "nativity":  "Native",
                                    "dialect":   info["dialect"],
                                    "save_path": None,
                                    "status":    "error",
                                    "error":     str(e)[:80],
                                    "cv_accent": info["cv_accent"],
                                    "up_votes":  info["up_votes"],
                                })
                                errors += 1

                        pbar.update(1)
                        # Stop early once we have everything
                        if extracted + errors + len(already_on_disk) >= len(clips_needed):
                            break

                pbar.close()
                stream_resp2.close()
                print(f"\nPass 2 complete — read {wrapper2.bytes_read / 1e6:.0f} MB of archive")

                print(f"\nCommon Voice extraction complete:")
                print(f"   Extracted       : {extracted}")
                print(f"   Already existed : {len(already_on_disk)}")
                print(f"   Errors          : {errors}")
                if cv_records:
                    b_df = pd.DataFrame(cv_records)
                    print(f"   Accent breakdown:")
                    print(b_df["cv_accent"].value_counts().to_string())
                else:
                    print("   WARNING: No clips recorded — check clips_prefix above")

    manifest_records.extend(cv_records)


# ════════════════════════════════════════════════════════════
# PART C — MASTER MANIFEST
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 58)
print("  PART C — Master Manifest")
print("=" * 58)

manifest_df = pd.DataFrame(manifest_records)

usable = manifest_df[
    manifest_df["status"].isin(["downloaded", "already_exists", "extracted"]) &
    manifest_df["save_path"].notna()
].copy()

usable["file_on_disk"] = usable["save_path"].apply(os.path.exists)
ghost = (~usable["file_on_disk"]).sum()
if ghost:
    print(f"Removing {ghost} manifest entries where file missing on disk")
usable = usable[usable["file_on_disk"]].drop(columns=["file_on_disk"])

n_native    = (usable["label"] == 1).sum()
n_nonnative = (usable["label"] == 0).sum()
ratio       = n_native / n_nonnative if n_nonnative > 0 else float("inf")

print(f"\nFinal manifest:")
print(f"   Total usable files : {len(usable)}")
print(f"\n   By source:")
print(usable["source"].value_counts().to_string())
print(f"\n   By class:")
print(usable["nativity"].value_counts().to_string())
print(f"\n   By dialect:")
print(usable["dialect"].value_counts().to_string())
print(f"\n   Class balance:")
print(f"     Native     : {n_native}")
print(f"     Non-Native : {n_nonnative}")
print(f"     Ratio      : {ratio:.1f}x")

manifest_df.to_csv(f"{OUTPUT_DIR}/stage2_manifest.csv", index=False)
print(f"\nSaved: {OUTPUT_DIR}/stage2_manifest.csv")
print("\n✅ Stage 2 complete.")
print("⏭️  Next: Run CELL_3_stage3.py")

  PART A — Downloading Renan Training Audio (160 files)


Renan files:   0%|          | 0/160 [00:00<?, ?it/s]


Renan download complete:
   Downloaded     : 160
   Already existed: 0
   Failed         : 0

  PART B — Common Voice 24.0 Arabic via API (streaming)
Requesting presigned download URL from Mozilla API...
✅ Got presigned URL (valid for ~15 min)

Pass 1: Streaming archive to extract train.tsv...
(Reads sequentially until TSV is found — stops early)
  Found TSV: cv-corpus-24.0-2025-12-05/ar/train.tsv (9.1 MB)
  Train clips in TSV: 28,870
  Columns: ['client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment']
  After quality filter: 20,100
  Using accent column: 'accents'

  Accent breakdown:
accent
unspecified    20100

  Sampled 200 clips across 1 accent groups
  Pass 1 complete — read 9 MB of archive
  Sample expected tar paths:
    cv-corpus-24.0-2025-12-05/ar/clips/common_voice_ar_24050927.mp3
    cv-corpus-24.0-2025-12-05/ar/clips/common_voice_ar_24056931.mp3
    cv-corpus-24.0-2025-12-05/ar

Extracting CV clips:   0%|          | 0/200 [00:00<?, ?it/s]


Pass 2 complete — read 2973 MB of archive

Common Voice extraction complete:
   Extracted       : 200
   Already existed : 0
   Errors          : 0
   Accent breakdown:
cv_accent
unspecified    200

  PART C — Master Manifest

Final manifest:
   Total usable files : 360

   By source:
source
mozilla_common_voice_24    200
renan                      160

   By class:
nativity
Native        314
Non-Native     46

   By dialect:
dialect
Arabic_MSA    200
Arabic_SA      63
Arabic_QA      56
Arabic_AE      38
Arabic_KW       3

   Class balance:
     Native     : 314
     Non-Native : 46
     Ratio      : 6.8x

Saved: /content/drive/MyDrive/team_databaes/outputs/stage2_manifest.csv

✅ Stage 2 complete.
⏭️  Next: Run CELL_3_stage3.py


In [20]:
# ============================================================
# CELL 3 — STAGE 3: PREPROCESSING
# Reads stage2_manifest.csv from Drive.
# Preprocessed WAVs saved directly to Drive.
#
# Pipeline per file: resample → RMS normalize → VAD trim
#                    → denoise → duration gate → save WAV
#
# RESUME SAFE: skips files whose output WAV already exists on
# Drive. If the session drops mid-run, re-run this cell and
# it will continue from where it stopped.
# ============================================================

import os, warnings
import numpy as np, pandas as pd
import librosa, soundfile as sf, noisereduce as nr
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

PROJECT_ROOT  = "/content/drive/MyDrive/team_databaes"
MANIFEST_IN   = f"{PROJECT_ROOT}/outputs/stage2_manifest.csv"
MANIFEST_OUT  = f"{PROJECT_ROOT}/outputs/stage3_manifest.csv"
PREPROC_DIR   = f"{PROJECT_ROOT}/data/preprocessed"

TARGET_SR    = 16000
TARGET_RMS   = 0.05
MIN_DURATION = 3.0
VAD_TOP_DB   = 20
NR_PROP      = 0.75

# ── Preprocessing functions ───────────────────────────────────

def load_audio(filepath):
    y, sr = librosa.load(filepath, sr=TARGET_SR, mono=True)
    return y, sr

def rms_normalize(y, target_rms=TARGET_RMS):
    current_rms = np.sqrt(np.mean(y ** 2))
    if current_rms < 1e-9:
        return y
    return y * (target_rms / current_rms)

def vad_trim_edges(y, top_db=VAD_TOP_DB):
    # Trims leading/trailing silence ONLY — internal pauses are preserved
    y_trimmed, _ = librosa.effects.trim(y, top_db=top_db)
    return y_trimmed

def reduce_noise(y, sr, prop_decrease=NR_PROP):
    return nr.reduce_noise(y=y, sr=sr, prop_decrease=prop_decrease, stationary=True)

def preprocess_file(filepath, label, dp_id):
    result = {
        "dp_id":           dp_id,
        "original_path":   filepath,
        "preproc_status":  None,
        "preproc_path":    None,
        "duration_before": None,
        "duration_after":  None,
        "dropped_reason":  None,
    }
    try:
        y, sr = load_audio(filepath)
        result["duration_before"] = round(len(y) / sr, 2)

        y = rms_normalize(y)
        y = vad_trim_edges(y)
        y = reduce_noise(y, sr)

        duration_after = len(y) / sr
        result["duration_after"] = round(duration_after, 2)

        if duration_after < MIN_DURATION:
            result["preproc_status"] = "dropped_too_short"
            result["dropped_reason"] = f"Only {duration_after:.1f}s after VAD trim"
            return result

        subdir   = "native" if label == 1 else "non_native"
        out_path = os.path.join(PREPROC_DIR, subdir, f"{dp_id}.wav")
        sf.write(out_path, y, TARGET_SR, subtype="PCM_16")

        result["preproc_status"] = "ok"
        result["preproc_path"]   = out_path

    except Exception as e:
        result["preproc_status"] = "error"
        result["dropped_reason"] = str(e)[:120]

    return result

# ── Load manifest ─────────────────────────────────────────────
print("=" * 58)
print("  STAGE 3 — PREPROCESSING")
print("=" * 58)

df = pd.read_csv(MANIFEST_IN)

processable = df[
    df["status"].isin(["downloaded", "already_exists", "extracted"]) &
    df["save_path"].notna()
].copy()

print(f"\nTotal files in manifest : {len(df)}")
print(f"Files to process        : {len(processable)}")
print(f"Target SR               : {TARGET_SR} Hz")
print(f"Target RMS              : {TARGET_RMS}")
print(f"VAD top_db              : {VAD_TOP_DB}")
print(f"Noise reduction         : {int(NR_PROP*100)}%")
print(f"Min duration gate       : {MIN_DURATION}s\n")

# ── RESUME: skip files whose output WAV already exists on Drive
def get_expected_out_path(row):
    subdir = "native" if row["label"] == 1 else "non_native"
    return os.path.join(PREPROC_DIR, subdir, f"{row['dp_id']}.wav")

processable["expected_out"] = processable.apply(get_expected_out_path, axis=1)
already_done = processable[processable["expected_out"].apply(os.path.exists)]
todo         = processable[~processable["expected_out"].apply(os.path.exists)]

if len(already_done) > 0:
    print(f"⏭️  Skipping {len(already_done)} files already preprocessed on Drive (resume)")
print(f"Processing {len(todo)} files...\n")

# ── Run preprocessing ─────────────────────────────────────────
preproc_results = []

# Add already-done files back as synthetic "ok" results
for _, row in already_done.iterrows():
    preproc_results.append({
        "dp_id":           row["dp_id"],
        "original_path":   row["save_path"],
        "preproc_status":  "ok",
        "preproc_path":    row["expected_out"],
        "duration_before": None,   # not re-measured, not needed
        "duration_after":  None,
        "dropped_reason":  None,
    })

# Process remaining files
for _, row in tqdm(todo.iterrows(), total=len(todo), desc="Preprocessing"):
    res = preprocess_file(
        filepath = row["save_path"],
        label    = row["label"],
        dp_id    = row["dp_id"],
    )
    preproc_results.append(res)

# ── Merge into manifest ───────────────────────────────────────
preproc_df = pd.DataFrame(preproc_results)
df_out     = processable.merge(preproc_df, on="dp_id", how="left")

ok      = df_out[df_out["preproc_status"] == "ok"]
dropped = df_out[df_out["preproc_status"] == "dropped_too_short"]
errors  = df_out[df_out["preproc_status"] == "error"]

# Measure durations for files that were freshly processed (not skipped)
# For skipped files, load duration from existing WAV
def get_duration(path):
    try:
        info = sf.info(path)
        return round(info.duration, 2)
    except Exception:
        return None

ok_with_dur = ok.copy()
mask_no_dur = ok_with_dur["duration_after"].isna()
if mask_no_dur.sum() > 0:
    ok_with_dur.loc[mask_no_dur, "duration_after"] = \
        ok_with_dur.loc[mask_no_dur, "preproc_path"].apply(get_duration)
    df_out.update(ok_with_dur[["dp_id","duration_after"]].set_index("dp_id"))

# Re-read ok after update
ok = df_out[df_out["preproc_status"] == "ok"]

# ── Report ────────────────────────────────────────────────────
print("\n" + "=" * 58)
print("  STAGE 3 — REPORT")
print("=" * 58)
print(f"\n   Successfully preprocessed : {len(ok)}")
print(f"   Dropped (too short)       : {len(dropped)}")
print(f"   Errors                    : {len(errors)}")

if len(dropped) > 0:
    print(f"\n   Dropped files:")
    cols = [c for c in ["dp_id","nativity","dialect","dropped_reason"] if c in dropped.columns]
    print(dropped[cols].to_string(index=False))

if len(errors) > 0:
    print(f"\n   Error files:")
    cols = [c for c in ["dp_id","nativity","dropped_reason"] if c in errors.columns]
    print(errors[cols].to_string(index=False))

ok_dur = ok["duration_after"].dropna()
if len(ok_dur) > 0:
    print(f"\n   Duration after VAD trim:")
    print(f"     mean={ok_dur.mean():.1f}s  min={ok_dur.min():.1f}s  max={ok_dur.max():.1f}s")

print(f"\n   Usable files by class:")
if "nativity" in ok.columns:
    print(ok["nativity"].value_counts().to_string())

est_segs = ok_dur.apply(lambda d: max(0, int((d - 3.0) / 1.0) + 1)).sum()
print(f"\n   Estimated 3s segments (Stage 4 preview): ~{int(est_segs):,}")
print(f"   (3s window, 1s hop, based on trimmed durations)")

# ── Save manifest ─────────────────────────────────────────────
df_out.to_csv(MANIFEST_OUT, index=False)
print(f"\nSaved: {MANIFEST_OUT}")
print("\n✅ Stage 3 complete.")
print("⏭️  Next: Run stage4_splitting.py")

  STAGE 3 — PREPROCESSING

Total files in manifest : 360
Files to process        : 360
Target SR               : 16000 Hz
Target RMS              : 0.05
VAD top_db              : 20
Noise reduction         : 75%
Min duration gate       : 3.0s

Processing 360 files...



Preprocessing:   0%|          | 0/360 [00:00<?, ?it/s]


  STAGE 3 — REPORT

   Successfully preprocessed : 213
   Dropped (too short)       : 147
   Errors                    : 0

   Dropped files:
    dp_id   nativity    dialect           dropped_reason
      529     Native  Arabic_QA Only 0.4s after VAD trim
      351     Native  Arabic_QA Only 0.1s after VAD trim
      407 Non-Native  Arabic_QA Only 0.3s after VAD trim
cv_000199     Native Arabic_MSA Only 2.0s after VAD trim
cv_000112     Native Arabic_MSA Only 1.4s after VAD trim
cv_000156     Native Arabic_MSA Only 1.8s after VAD trim
cv_000014     Native Arabic_MSA Only 2.3s after VAD trim
cv_000138     Native Arabic_MSA Only 1.4s after VAD trim
cv_000056     Native Arabic_MSA Only 2.2s after VAD trim
cv_000189     Native Arabic_MSA Only 1.1s after VAD trim
cv_000064     Native Arabic_MSA Only 1.9s after VAD trim
cv_000186     Native Arabic_MSA Only 1.4s after VAD trim
cv_000088     Native Arabic_MSA Only 2.4s after VAD trim
cv_000027     Native Arabic_MSA Only 2.9s after VAD trim
cv

In [21]:
# ============================================================
# STAGE 4: AUDIO SPLITTING
# Cuts every preprocessed WAV into overlapping 3s segments
# with a 1s hop. Each segment inherits parent file's label.
#
# COLAB SETUP (run this cell first in your notebook):
#   from google.colab import drive
#   drive.mount('/content/drive')
# ============================================================
import os
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm.notebook import tqdm   # notebook-friendly progress bars

# ── Google Drive path config ─────────────────────────────────────
# Change this to match your actual Drive folder name
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

MANIFEST_IN  = f"{PROJECT_ROOT}/outputs/stage3_manifest.csv"
MANIFEST_OUT = f"{PROJECT_ROOT}/outputs/stage4_manifest.csv"
SEGMENTS_DIR = f"{PROJECT_ROOT}/data/segments"

WINDOW_SEC = 3.0
HOP_SEC    = 1.0
TARGET_SR  = 16000

os.makedirs(f"{SEGMENTS_DIR}/native",     exist_ok=True)
os.makedirs(f"{SEGMENTS_DIR}/non_native", exist_ok=True)

df     = pd.read_csv(MANIFEST_IN)
usable = df[df["preproc_status"] == "ok"].copy()
print(f"Usable files: {len(usable)}")

window_samples = int(WINDOW_SEC * TARGET_SR)   # 48000 samples
hop_samples    = int(HOP_SEC    * TARGET_SR)   # 16000 samples

segment_records = []

for _, row in tqdm(usable.iterrows(), total=len(usable), desc="Splitting"):
    try:
        y, sr = librosa.load(row["preproc_path"], sr=TARGET_SR, mono=True)
    except Exception as e:
        print(f"  Load error {row['dp_id']}: {e}")
        continue

    n_segments = max(0, (len(y) - window_samples) // hop_samples + 1)
    subdir     = "native" if row["label"] == 1 else "non_native"

    for i in range(n_segments):
        start   = i * hop_samples
        end     = start + window_samples
        segment = y[start:end]

        if len(segment) < window_samples:
            segment = np.pad(segment, (0, window_samples - len(segment)))

        seg_id   = f"{row['dp_id']}_seg{i:04d}"
        seg_path = os.path.join(SEGMENTS_DIR, subdir, f"{seg_id}.wav")

        sf.write(seg_path, segment, TARGET_SR, subtype="PCM_16")

        segment_records.append({
            "seg_id":    seg_id,
            "parent_id": row["dp_id"],
            "seg_index": i,
            "label":     row["label"],
            "nativity":  row["nativity"],
            "dialect":   row["dialect"],
            "source":    row["source"],
            "seg_path":  seg_path,
            "start_sec": round(i * HOP_SEC, 2),
            "end_sec":   round(i * HOP_SEC + WINDOW_SEC, 2),
        })

seg_df = pd.DataFrame(segment_records)
seg_df.to_csv(MANIFEST_OUT, index=False)

print(f"\nTotal segments created : {len(seg_df)}")
print(f"Native segments        : {(seg_df['label']==1).sum()}")
print(f"Non-native segments    : {(seg_df['label']==0).sum()}")
print(f"Saved: {MANIFEST_OUT}")

Usable files: 213


Splitting:   0%|          | 0/213 [00:00<?, ?it/s]


Total segments created : 6867
Native segments        : 4719
Non-native segments    : 2148
Saved: /content/drive/MyDrive/team_databaes/outputs/stage4_manifest.csv


In [22]:
# ============================================================
# STAGE 5: DATA AUGMENTATION (training segments only)
# Non-native (minority): 5 augmentations
# Native (majority):     1 augmentation (time stretch only)
# NEVER augment test data.
#
# COLAB NOTE: Augmentation is CPU-heavy. On Colab free tier
# (~17k segments × 5 augs) expect ~45–90 min. Colab Pro/T4
# does not speed this up — librosa runs on CPU only.
# Consider running overnight or in sections using RESUME below.
# ============================================================
import os
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm.notebook import tqdm

# ── Google Drive path config ─────────────────────────────────────
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

MANIFEST_IN  = f"{PROJECT_ROOT}/outputs/stage4_manifest.csv"
MANIFEST_OUT = f"{PROJECT_ROOT}/outputs/stage5_manifest.csv"
AUG_DIR      = f"{PROJECT_ROOT}/data/augmented"
TARGET_SR    = 16000
RANDOM_SEED  = 42
np.random.seed(RANDOM_SEED)

os.makedirs(f"{AUG_DIR}/native",     exist_ok=True)
os.makedirs(f"{AUG_DIR}/non_native", exist_ok=True)

seg_df = pd.read_csv(MANIFEST_IN)
print(f"Segments to augment: {len(seg_df)}")

# ── RESUME SUPPORT ───────────────────────────────────────────────
# If the Colab session disconnects mid-run, re-running will skip
# already-completed segments instead of starting over.
already_done = set()
if os.path.exists(MANIFEST_OUT):
    done_df      = pd.read_csv(MANIFEST_OUT)
    already_done = set(done_df[done_df["aug_type"] != "original"]["orig_seg"].dropna())
    print(f"Resuming — {len(already_done)} segments already augmented, skipping.")

# ── Augmentation functions ───────────────────────────────────────
def time_stretch(y, rate):
    return librosa.effects.time_stretch(y, rate=rate)

def pitch_shift(y, sr, steps):
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=steps)

def add_noise(y, sigma=0.005):
    noise = np.random.randn(len(y)) * sigma
    return np.clip(y + noise, -1.0, 1.0)

def save_augmented(y, sr, seg_id, aug_name, label):
    subdir   = "native" if label == 1 else "non_native"
    filename = f"{seg_id}_{aug_name}.wav"
    out_path = os.path.join(AUG_DIR, subdir, filename)
    sf.write(out_path, y, sr, subtype="PCM_16")
    return out_path

# ── Apply augmentations ──────────────────────────────────────────
aug_records = []

for _, row in tqdm(seg_df.iterrows(), total=len(seg_df), desc="Augmenting"):
    # Skip if already processed (resume support)
    if row["seg_id"] in already_done:
        continue

    try:
        y, sr = librosa.load(row["seg_path"], sr=TARGET_SR, mono=True)
    except Exception as e:
        print(f"  Load error {row['seg_id']}: {e}")
        continue

    augmentations = []
    if row["label"] == 0:
        # Non-native (minority class): all 3 augmentation types (5 variants)
        augmentations = [
            ("ts_slow",  time_stretch(y, 0.9)),
            ("ts_fast",  time_stretch(y, 1.1)),
            ("ps_up",    pitch_shift(y, sr, +2)),
            ("ps_down",  pitch_shift(y, sr, -2)),
            ("noise",    add_noise(y)),
        ]
    else:
        # Native (majority class): time stretch only
        augmentations = [
            ("ts_slow",  time_stretch(y, 0.9)),
        ]

    for aug_name, y_aug in augmentations:
        if len(y_aug) > 48000:
            y_aug = y_aug[:48000]
        elif len(y_aug) < 48000:
            y_aug = np.pad(y_aug, (0, 48000 - len(y_aug)))

        out_path = save_augmented(y_aug, sr, row["seg_id"], aug_name, row["label"])
        aug_records.append({
            "seg_id":    f"{row['seg_id']}_{aug_name}",
            "parent_id": row["parent_id"],
            "seg_index": row["seg_index"],
            "label":     row["label"],
            "nativity":  row["nativity"],
            "dialect":   row["dialect"],
            "source":    "augmented",
            "seg_path":  out_path,
            "aug_type":  aug_name,
            "orig_seg":  row["seg_id"],
        })

# ── Combine originals + new augmentations ────────────────────────
aug_df = pd.DataFrame(aug_records)

# If resuming, load existing aug records and append only new ones
if os.path.exists(MANIFEST_OUT) and len(already_done) > 0:
    existing = pd.read_csv(MANIFEST_OUT)
    aug_df   = pd.concat([existing, aug_df], ignore_index=True)
    # Rebuild: originals are from stage4_manifest, augs from aug_df
    originals = seg_df.assign(aug_type="original", orig_seg=None)
    all_segs  = pd.concat(
        [originals, aug_df[aug_df["aug_type"] != "original"]],
        ignore_index=True
    )
else:
    originals = seg_df.assign(aug_type="original", orig_seg=None)
    all_segs  = pd.concat([originals, aug_df], ignore_index=True)

all_segs.to_csv(MANIFEST_OUT, index=False)

print(f"\nOriginal segments  : {len(seg_df)}")
print(f"Augmented segments : {(all_segs['aug_type'] != 'original').sum()}")
print(f"Total segments     : {len(all_segs)}")
print(f"Native total       : {(all_segs['label']==1).sum()}")
print(f"Non-native total   : {(all_segs['label']==0).sum()}")
print(f"Saved: {MANIFEST_OUT}")

Segments to augment: 6867


Augmenting:   0%|          | 0/6867 [00:00<?, ?it/s]


Original segments  : 6867
Augmented segments : 15459
Total segments     : 22326
Native total       : 9438
Non-native total   : 12888
Saved: /content/drive/MyDrive/team_databaes/outputs/stage5_manifest.csv


In [24]:
# ============================================================
# STAGE 6A: wav2vec2 EMBEDDING EXTRACTION
# Model: jonatasgrosman/wav2vec2-large-xlsr-53-arabic (frozen)
# Output: (N_segments, 768) numpy array via mean pooling
#
# COLAB NOTES:
#   - Runtime > GPU is STRONGLY recommended (Runtime → Change runtime type)
#   - Model downloads ~1.2GB on first run, cached in /root/.cache/
#     BUT Colab cache clears on session reset — model re-downloads each new session
#     To persist the cache: set HF_HOME to a Drive path (see config below)
#   - BATCH_SIZE=16 works on T4 (16GB VRAM). Use 8 on free tier (12GB)
#   - Checkpoint saves every CHECKPOINT_EVERY batches so a session
#     disconnect doesn't lose all progress
# ============================================================
import os
import numpy as np
import pandas as pd
import torch
import librosa
from transformers import Wav2Vec2Model, Wav2Vec2Processor
from tqdm.notebook import tqdm

# ── Google Drive path config ─────────────────────────────────────
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

MANIFEST_IN       = f"{PROJECT_ROOT}/outputs/stage5_manifest.csv"
EMBED_DIR         = f"{PROJECT_ROOT}/data/embeddings"
TARGET_SR         = 16000
BATCH_SIZE        = 16        # reduce to 8 if CUDA OOM (free tier Colab)
CHECKPOINT_EVERY  = 50        # save partial results every N batches
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"

# ── Persist HuggingFace model cache to Drive (avoids re-downloading each session)
HF_CACHE = f"{PROJECT_ROOT}/hf_cache"
os.environ["HF_HOME"]            = HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE
os.makedirs(HF_CACHE,  exist_ok=True)
os.makedirs(EMBED_DIR, exist_ok=True)

print(f"Using device: {DEVICE}")
if DEVICE == "cpu":
    print("WARNING: No GPU detected. Embedding extraction will be very slow on CPU.")
    print("Go to Runtime → Change runtime type → GPU (T4)")

# ── Load model ───────────────────────────────────────────────────
print("Loading wav2vec2 model (downloads ~1.2GB if not cached)...")
processor = Wav2Vec2Processor.from_pretrained("jonatasgrosman/wav2vec2-large-xlsr-53-arabic")
model     = Wav2Vec2Model.from_pretrained("jonatasgrosman/wav2vec2-large-xlsr-53-arabic")
model     = model.to(DEVICE)
model.eval()
print("Model loaded.")

seg_df = pd.read_csv(MANIFEST_IN)
print(f"Total segments to embed: {len(seg_df)}")

# ── CHECKPOINT / RESUME ──────────────────────────────────────────
checkpoint_embeds_path  = f"{EMBED_DIR}/wav2vec2_embeddings_partial.npy"
checkpoint_segids_path  = f"{EMBED_DIR}/wav2vec2_seg_ids_partial.npy"
start_batch = 0

all_embeddings = []
all_seg_ids    = []

if os.path.exists(checkpoint_embeds_path) and os.path.exists(checkpoint_segids_path):
    prev_embeds  = np.load(checkpoint_embeds_path)
    prev_seg_ids = np.load(checkpoint_segids_path, allow_pickle=True)
    all_embeddings.append(prev_embeds)
    all_seg_ids.extend(prev_seg_ids.tolist())
    start_batch  = (len(prev_seg_ids) // BATCH_SIZE)
    print(f"Resuming from batch {start_batch} ({len(prev_seg_ids)} segments already done)")

# ── Embedding extraction ─────────────────────────────────────────
def extract_embedding_batch(paths):
    waveforms = []
    for path in paths:
        y, _ = librosa.load(path, sr=TARGET_SR, mono=True)
        if len(y) > 48000:
            y = y[:48000]
        elif len(y) < 48000:
            y = np.pad(y, (0, 48000 - len(y)))
        waveforms.append(y)

    inputs = processor(
        waveforms,
        sampling_rate=TARGET_SR,
        return_tensors="pt",
        padding=True
    )
    input_values = inputs.input_values.to(DEVICE)

    with torch.no_grad():
        outputs    = model(input_values)
        embeddings = outputs.last_hidden_state.mean(dim=1)   # (batch, 768)

    return embeddings.cpu().numpy()

paths   = seg_df["seg_path"].tolist()
seg_ids = seg_df["seg_id"].tolist()
errors  = []

# Skip already-processed batches
paths_todo   = paths[start_batch * BATCH_SIZE:]
seg_ids_todo = seg_ids[start_batch * BATCH_SIZE:]

for i in tqdm(range(0, len(paths_todo), BATCH_SIZE), desc="Extracting embeddings"):
    batch_paths   = paths_todo[i : i + BATCH_SIZE]
    batch_seg_ids = seg_ids_todo[i : i + BATCH_SIZE]

    try:
        embeds = extract_embedding_batch(batch_paths)
        all_embeddings.append(embeds)
        all_seg_ids.extend(batch_seg_ids)
    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            print(f"\nOOM at batch {i} — try reducing BATCH_SIZE to {BATCH_SIZE // 2}")
            torch.cuda.empty_cache()
            raise
        print(f"  Batch {i} error: {e}")
        errors.append({"batch_start": i, "error": str(e)})
        all_embeddings.append(np.zeros((len(batch_paths), 768)))
        all_seg_ids.extend(batch_seg_ids)

    # Save checkpoint periodically (protects against Colab session drops)
    if (i // BATCH_SIZE + 1) % CHECKPOINT_EVERY == 0:
        partial_embeds = np.vstack(all_embeddings)
        np.save(checkpoint_embeds_path, partial_embeds)
        np.save(checkpoint_segids_path, np.array(all_seg_ids))
        print(f"  Checkpoint saved at batch {i // BATCH_SIZE + 1} ({len(all_seg_ids)} segments)")

# ── Final save ───────────────────────────────────────────────────
X_wav2vec = np.vstack(all_embeddings)   # (N_segments, 768)
np.save(f"{EMBED_DIR}/wav2vec2_embeddings.npy", X_wav2vec)
np.save(f"{EMBED_DIR}/wav2vec2_seg_ids.npy",    np.array(all_seg_ids))

# Clean up checkpoint files
for f in [checkpoint_embeds_path, checkpoint_segids_path]:
    if os.path.exists(f):
        os.remove(f)

print(f"\nEmbeddings shape : {X_wav2vec.shape}")
print(f"Saved: {EMBED_DIR}/wav2vec2_embeddings.npy")
if errors:
    print(f"WARNING: Errors in {len(errors)} batches")

Using device: cuda
Loading wav2vec2 model (downloads ~1.2GB if not cached)...


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: jonatasgrosman/wav2vec2-large-xlsr-53-arabic
Key            | Status     |  | 
---------------+------------+--+-
lm_head.bias   | UNEXPECTED |  | 
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.
Total segments to embed: 22326


Extracting embeddings:   0%|          | 0/1396 [00:00<?, ?it/s]

  Checkpoint saved at batch 50 (800 segments)
  Checkpoint saved at batch 100 (1600 segments)
  Checkpoint saved at batch 150 (2400 segments)
  Checkpoint saved at batch 200 (3200 segments)
  Checkpoint saved at batch 250 (4000 segments)
  Checkpoint saved at batch 300 (4800 segments)
  Checkpoint saved at batch 350 (5600 segments)
  Checkpoint saved at batch 400 (6400 segments)
  Checkpoint saved at batch 450 (7200 segments)
  Checkpoint saved at batch 500 (8000 segments)
  Checkpoint saved at batch 550 (8800 segments)
  Checkpoint saved at batch 600 (9600 segments)
  Checkpoint saved at batch 650 (10400 segments)
  Checkpoint saved at batch 700 (11200 segments)
  Checkpoint saved at batch 750 (12000 segments)
  Checkpoint saved at batch 800 (12800 segments)
  Checkpoint saved at batch 850 (13600 segments)
  Checkpoint saved at batch 900 (14400 segments)
  Checkpoint saved at batch 950 (15200 segments)
  Checkpoint saved at batch 1000 (16000 segments)
  Checkpoint saved at batch 1050 

In [26]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Loaded parameters: {total_params:,}")
# Expect ~300M for wav2vec2-large — confirms full encoder loaded correctly

Loaded parameters: 315,438,720


In [29]:
# ============================================================
# STAGE 6B: PROSODIC FEATURE EXTRACTION
# Runs on UTTERANCE level (stage3_manifest).
# All segments from the same recording share the same 6-dim
# prosodic vector (looked up by parent_id in Stage 7).
#
# COLAB NOTES:
#   - parselmouth must be installed: pip install praat-parselmouth
#   - This runs on CPU — no GPU benefit here
#   - Can be run in parallel with Stage 6A (different inputs/outputs)
# ============================================================
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# Install parselmouth if not already installed (safe to re-run)
try:
    import parselmouth
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "praat-parselmouth", "-q"], check=True)
    import parselmouth

# ── Google Drive path config ─────────────────────────────────────
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

MANIFEST_IN  = f"{PROJECT_ROOT}/outputs/stage3_manifest.csv"
PROSODIC_OUT = f"{PROJECT_ROOT}/data/embeddings/prosodic_features.csv"

os.makedirs(f"{PROJECT_ROOT}/data/embeddings", exist_ok=True)


def extract_prosodic_features(audio_path):
    """
    Extract 6 prosodic features from a full utterance WAV file.
    Returns np.array of shape (6,) or zeros on failure.
    Features: [f0_mean, f0_var, speaking_rate, pause_freq,
               pause_dur_mean, npvi]
    """
    try:
        sound    = parselmouth.Sound(audio_path)
        duration = sound.end_time - sound.start_time

        # ── F0 mean and variance ──────────────────────────────
        pitch     = sound.to_pitch(time_step=0.01, pitch_floor=75, pitch_ceiling=600)
        f0_vals   = pitch.selected_array['frequency']
        f0_voiced = f0_vals[f0_vals > 0]

        f0_mean = float(np.mean(f0_voiced)) if len(f0_voiced) > 0 else 0.0
        f0_var  = float(np.var(f0_voiced))  if len(f0_voiced) > 0 else 0.0

        # ── Speaking rate (syllable-proxy peaks/sec) ──────────
        intensity  = sound.to_intensity(minimum_pitch=75)
        int_values = intensity.values.flatten()
        int_mean   = float(np.mean(int_values))

        peaks = 0
        for j in range(1, len(int_values) - 1):
            if (int_values[j] > int_mean and
                    int_values[j] > int_values[j-1] and
                    int_values[j] > int_values[j+1]):
                peaks += 1

        speaking_rate = peaks / duration if duration > 0 else 0.0

        # ── Pause frequency and mean pause duration ───────────
        pause_threshold_sec = 0.2

        f0_times  = [pitch.get_time_from_frame_number(i+1)
                     for i in range(pitch.get_number_of_frames())]
        is_voiced = [pitch.get_value_at_time(t) is not None and
                     pitch.get_value_at_time(t) > 0
                     for t in f0_times]

        pauses      = []
        in_pause    = False
        pause_start = 0.0

        for t, voiced in zip(f0_times, is_voiced):
            if not voiced and not in_pause:
                in_pause    = True
                pause_start = t
            elif voiced and in_pause:
                pause_dur = t - pause_start
                if pause_dur >= pause_threshold_sec:
                    pauses.append(pause_dur)
                in_pause = False

        pause_freq     = len(pauses) / duration if duration > 0 else 0.0
        pause_dur_mean = float(np.mean(pauses)) if pauses else 0.0

        # ── nPVI ──────────────────────────────────────────────
        vowel_durations = []
        in_vowel        = False
        vowel_start     = 0.0

        for t, voiced in zip(f0_times, is_voiced):
            if voiced and not in_vowel:
                in_vowel    = True
                vowel_start = t
            elif not voiced and in_vowel:
                vowel_dur = t - vowel_start
                if vowel_dur >= 0.03:
                    vowel_durations.append(vowel_dur)
                in_vowel = False

        if len(vowel_durations) >= 2:
            durs  = np.array(vowel_durations)
            dk    = durs[:-1]
            dk1   = durs[1:]
            denom = (dk + dk1) / 2
            valid = denom > 0
            npvi  = float(np.mean(np.abs(dk[valid] - dk1[valid]) / denom[valid]) * 100)
        else:
            npvi = 0.0

        return np.array([f0_mean, f0_var, speaking_rate,
                         pause_freq, pause_dur_mean, npvi], dtype=np.float32)

    except Exception as e:
        print(f"  Prosodic extraction failed for {audio_path}: {e}")
        return np.zeros(6, dtype=np.float32)


# ── Run extraction ───────────────────────────────────────────────
df     = pd.read_csv(MANIFEST_IN)
usable = df[df["preproc_status"] == "ok"].copy()
print(f"Extracting prosodic features from {len(usable)} utterances...")

records = []
for _, row in tqdm(usable.iterrows(), total=len(usable), desc="Prosodic features"):
    features = extract_prosodic_features(row["preproc_path"])
    records.append({
        "dp_id":          row["dp_id"],
        "label":          row["label"],
        "f0_mean":        features[0],
        "f0_var":         features[1],
        "speaking_rate":  features[2],
        "pause_freq":     features[3],
        "pause_dur_mean": features[4],
        "npvi":           features[5],
    })

prosodic_df = pd.DataFrame(records)
prosodic_df.to_csv(PROSODIC_OUT, index=False)

print(f"\nProsodic features shape : {prosodic_df.shape}")
print(f"Saved: {PROSODIC_OUT}")

print("\nFeature means by class:")
print(prosodic_df.groupby("label")[
    ["f0_mean","f0_var","speaking_rate","pause_freq","pause_dur_mean","npvi"]
].mean().round(3).to_string())

Extracting prosodic features from 213 utterances...


Prosodic features:   0%|          | 0/213 [00:00<?, ?it/s]


Prosodic features shape : (213, 8)
Saved: /content/drive/MyDrive/team_databaes/data/embeddings/prosodic_features.csv

Feature means by class:
          f0_mean       f0_var  speaking_rate  pause_freq  pause_dur_mean       npvi
label                                                                               
0      197.574005  3236.242920          6.613       0.485           0.570  78.156998
1      203.115997  2822.447021          6.764       0.405           0.415  82.739998


In [30]:
# ============================================================
# STAGE 7: FEATURE FUSION
# Concatenates wav2vec2 (768-dim) + prosodic (6-dim) per segment
# → 774-dim feature vector.
# StandardScaler fitted on TRAIN only, applied to test.
# Group-level split by parent_id — no data leakage.
#
# COLAB NOTES:
#   - Pure numpy/sklearn — no GPU needed, runs fast on CPU
#   - Loads .npy files from Drive (may be slow if files are large;
#     consider copying to /content/ first for faster I/O)
# ============================================================
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
import joblib

# ── Google Drive path config ─────────────────────────────────────
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

EMBED_DIR    = f"{PROJECT_ROOT}/data/embeddings"
MANIFEST_IN  = f"{PROJECT_ROOT}/outputs/stage5_manifest.csv"
FEATURES_OUT = f"{PROJECT_ROOT}/outputs/stage7_features.npz"

os.makedirs(f"{PROJECT_ROOT}/outputs", exist_ok=True)

# ── Optional: copy .npy files to local /content/ for faster I/O ──
# Uncomment if loading from Drive is very slow (large files)
# import shutil
# shutil.copy(f"{EMBED_DIR}/wav2vec2_embeddings.npy", "/content/wav2vec2_embeddings.npy")
# shutil.copy(f"{EMBED_DIR}/wav2vec2_seg_ids.npy",    "/content/wav2vec2_seg_ids.npy")
# X_wav2vec   = np.load("/content/wav2vec2_embeddings.npy")
# seg_ids_w2v = np.load("/content/wav2vec2_seg_ids.npy", allow_pickle=True)

# ── Load wav2vec2 embeddings ─────────────────────────────────────
print("Loading wav2vec2 embeddings...")
X_wav2vec   = np.load(f"{EMBED_DIR}/wav2vec2_embeddings.npy")
seg_ids_w2v = np.load(f"{EMBED_DIR}/wav2vec2_seg_ids.npy", allow_pickle=True)
print(f"  Embeddings shape: {X_wav2vec.shape}")

# ── Load prosodic features ────────────────────────────────────────
print("Loading prosodic features...")
prosodic_df  = pd.read_csv(f"{EMBED_DIR}/prosodic_features.csv")
prosodic_map = prosodic_df.set_index("dp_id")[
    ["f0_mean","f0_var","speaking_rate","pause_freq","pause_dur_mean","npvi"]
].to_dict("index")
print(f"  Prosodic utterances: {len(prosodic_map)}")

# ── Load segment manifest ─────────────────────────────────────────
seg_df = pd.read_csv(MANIFEST_IN)
print(f"  Total segments in manifest: {len(seg_df)}")

seg_id_to_idx = {sid: i for i, sid in enumerate(seg_ids_w2v)}

fused_features = []
labels         = []
parent_ids     = []
seg_ids_final  = []
missing_w2v    = 0
missing_pros   = 0

for _, row in seg_df.iterrows():
    seg_id = row["seg_id"]

    if seg_id not in seg_id_to_idx:
        missing_w2v += 1
        continue

    w2v_embed   = X_wav2vec[seg_id_to_idx[seg_id]]   # (768,)
    parent_id   = row["parent_id"]
    base_parent = parent_id.split("_aug")[0] if "_aug" in parent_id else parent_id

    if base_parent in prosodic_map:
        pros_vec = np.array(list(prosodic_map[base_parent].values()), dtype=np.float32)
    else:
        pros_vec = np.zeros(6, dtype=np.float32)
        missing_pros += 1

    fused_features.append(np.concatenate([w2v_embed, pros_vec]))  # (774,)
    labels.append(row["label"])
    parent_ids.append(parent_id)
    seg_ids_final.append(seg_id)

X      = np.array(fused_features)
y      = np.array(labels)
groups = np.array(parent_ids)

print(f"\nFused feature matrix shape : {X.shape}")
print(f"Label distribution — Native: {(y==1).sum()}, Non-Native: {(y==0).sum()}")
if missing_w2v:
    print(f"WARNING: {missing_w2v} segments skipped (missing wav2vec2 embedding)")
if missing_pros:
    print(f"WARNING: {missing_pros} segments used zero prosodic vector")

# ── Group-level Train/Test Split ──────────────────────────────────
# Split by parent_id so no recording appears in both train and test
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"\nTrain segments : {X_train.shape[0]}")
print(f"Test segments  : {X_test.shape[0]}")
print(f"Train — Native: {(y_train==1).sum()}, Non-Native: {(y_train==0).sum()}")
print(f"Test  — Native: {(y_test==1).sum()},  Non-Native: {(y_test==0).sum()}")

# ── Normalize — fit on train only ────────────────────────────────
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)    # transform only — no refit

# ── Save ──────────────────────────────────────────────────────────
np.savez(
    FEATURES_OUT,
    X_train=X_train_sc,
    X_test=X_test_sc,
    y_train=y_train,
    y_test=y_test,
    train_idx=train_idx,
    test_idx=test_idx,
)
joblib.dump(scaler, f"{PROJECT_ROOT}/outputs/scaler.joblib")

print(f"\nFeatures saved : {FEATURES_OUT}")
print(f"Scaler saved   : {PROJECT_ROOT}/outputs/scaler.joblib")

Loading wav2vec2 embeddings...
  Embeddings shape: (22326, 1024)
Loading prosodic features...
  Prosodic utterances: 213
  Total segments in manifest: 22326

Fused feature matrix shape : (22326, 1030)
Label distribution — Native: 9438, Non-Native: 12888

Train segments : 20014
Test segments  : 2312
Train — Native: 8248, Non-Native: 11766
Test  — Native: 1190,  Non-Native: 1122

Features saved : /content/drive/MyDrive/team_databaes/outputs/stage7_features.npz
Scaler saved   : /content/drive/MyDrive/team_databaes/outputs/scaler.joblib


In [ ]:
# ============================================================
# STAGE 8: CLASSIFICATION
# Primary:  LinearSVC (liblinear) — fast on large datasets
#           RBF SVM on a 5,000-sample subset for comparison
# Baseline: Logistic Regression
#
# WHY LinearSVC instead of RBF SVM GridSearchCV?
#   RBF SVM is O(n²–n³) in training time.
#   With 20,000 segments it takes 2–3 hrs on Colab CPU.
#   LinearSVC is O(n) — same accuracy on high-dim data
#   (1030 features >> samples per class, linear boundary works well)
#   Typical finish time: 3–8 min on Colab free tier.
#
# COLAB NOTES:
#   - All CPU — GPU gives no speedup for SVM/LinearSVC
#   - n_jobs=-1 uses all available cores
#   - Model saved to Drive immediately after training
# ============================================================
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.svm import LinearSVC, SVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from scipy.stats import loguniform

# ── Google Drive path config ──────────────────────────────────
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"
FEATURES_IN  = f"{PROJECT_ROOT}/outputs/stage7_features.npz"
os.makedirs(f"{PROJECT_ROOT}/outputs", exist_ok=True)

# ── Load features ─────────────────────────────────────────────
print("Loading features from Drive...")
data    = np.load(FEATURES_IN)
X_train = data["X_train"]
X_test  = data["X_test"]
y_train = data["y_train"]
y_test  = data["y_test"]

print(f"Training on {X_train.shape[0]} segments, {X_train.shape[1]} features")
print(f"Test set    : {X_test.shape[0]} segments")
print(f"Train — Native: {(y_train==1).sum()}, Non-Native: {(y_train==0).sum()}")

# ── SKIP IF ALREADY TRAINED ───────────────────────────────────
_model_path = f"{PROJECT_ROOT}/outputs/svm_model.joblib"
if os.path.exists(_model_path):
    best_model = joblib.load(_model_path)
    print(f"\n⏭️  svm_model.joblib already exists — skipping training.")
    y_pred_test  = best_model.predict(X_test)
    y_pred_train = best_model.predict(X_train)
    print(f"Train F1 : {f1_score(y_train, y_pred_train):.4f}")
    print(f"Test F1  : {f1_score(y_test,  y_pred_test):.4f}")
    print(classification_report(y_test, y_pred_test,
                                 target_names=["Non-Native", "Native"]))
    print("Delete outputs/svm_model.joblib to force re-training.")
    raise SystemExit(0)

# ════════════════════════════════════════════════════════════
# PRIMARY: LinearSVC with RandomizedSearchCV
# LinearSVC is ~50x faster than RBF SVM on large datasets.
# On high-dimensional data (1030 features) linear kernels
# perform comparably to RBF — the data is already well-
# separated in high-dim space by wav2vec2 representations.
# ════════════════════════════════════════════════════════════
print("\n" + "="*55)
print("  PRIMARY — LinearSVC (fast, ~3–8 min)")
print("="*55)

param_dist = {
    "estimator__C": loguniform(1e-2, 1e2)   # log-uniform sample from 0.01 to 100
}

base_svc = LinearSVC(
    class_weight="balanced",
    max_iter=2000,
    random_state=42,
)

# CalibratedClassifierCV wraps LinearSVC to add predict_proba()
# (needed for Stage 9 ROC/AUC and Stage 10 confidence scores)
calibrated_svc = CalibratedClassifierCV(base_svc, cv=3)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    calibrated_svc,
    param_distributions=param_dist,
    n_iter=10,          # 10 random C values × 3-fold = 30 fits
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    verbose=2,
    random_state=42,
    refit=True,
)

print("Running RandomizedSearchCV (10 iterations × 3-fold = 30 fits)...")
print("Estimated time: 3–8 min\n")
search.fit(X_train, y_train)

best_model   = search.best_estimator_
best_C       = search.best_params_["estimator__C"]
best_cv_f1   = search.best_score_

print(f"\nBest C      : {best_C:.4f}")
print(f"Best CV F1  : {best_cv_f1:.4f}")

y_pred_train = best_model.predict(X_train)
y_pred_test  = best_model.predict(X_test)

print(f"\nTrain F1 : {f1_score(y_train, y_pred_train):.4f}")
print(f"Test F1  : {f1_score(y_test,  y_pred_test):.4f}")
print("\nClassification report (test set):")
print(classification_report(y_test, y_pred_test,
                             target_names=["Non-Native", "Native"]))

# ════════════════════════════════════════════════════════════
# COMPARISON: RBF SVM on a 5,000-sample subset
# Gives a fair RBF vs Linear comparison without 2hr wait.
# Uses stratified sampling to preserve class ratio.
# ════════════════════════════════════════════════════════════
print("\n" + "="*55)
print("  COMPARISON — RBF SVM on 5,000-sample subset")
print("="*55)

from sklearn.model_selection import StratifiedShuffleSplit

sss = StratifiedShuffleSplit(n_splits=1, train_size=5000, random_state=42)
sub_idx, _ = next(sss.split(X_train, y_train))
X_sub = X_train[sub_idx]
y_sub = y_train[sub_idx]

rbf_param_grid = {"C": [1, 10], "gamma": ["scale"]}
rbf_svm = SVC(kernel="rbf", class_weight="balanced",
               probability=True, random_state=42)
rbf_cv  = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

from sklearn.model_selection import GridSearchCV
rbf_search = GridSearchCV(rbf_svm, rbf_param_grid, cv=rbf_cv,
                           scoring="f1", n_jobs=-1, verbose=1)

print("Fitting RBF SVM on 5,000-sample subset (2×2 grid, 3-fold)...")
print("Estimated time: 2–5 min\n")
rbf_search.fit(X_sub, y_sub)

rbf_best = rbf_search.best_estimator_
rbf_f1   = f1_score(y_test, rbf_best.predict(X_test))
lin_f1   = f1_score(y_test, y_pred_test)

print(f"\nLinearSVC Test F1 (full train set) : {lin_f1:.4f}")
print(f"RBF SVM Test F1  (5k subset)       : {rbf_f1:.4f}")
print(f"Difference                         : {lin_f1 - rbf_f1:+.4f}")

if lin_f1 >= rbf_f1:
    print("→ LinearSVC wins or ties — keeping LinearSVC as primary model")
    final_model = best_model
    model_note  = "LinearSVC (CalibratedClassifierCV)"
else:
    print("→ RBF SVM wins — saving RBF as primary model")
    final_model = rbf_best
    model_note  = "RBF SVM (trained on 5k subset)"

# ════════════════════════════════════════════════════════════
# BASELINE: Logistic Regression
# ════════════════════════════════════════════════════════════
print("\n" + "="*55)
print("  BASELINE — Logistic Regression")
print("="*55)

lr = LogisticRegression(class_weight="balanced", max_iter=1000,
                         solver="saga", n_jobs=-1, random_state=42)
lr.fit(X_train, y_train)
lr_f1 = f1_score(y_test, lr.predict(X_test))

print(f"Logistic Regression Test F1 : {lr_f1:.4f}")
print(f"Primary model Test F1       : {f1_score(y_test, final_model.predict(X_test)):.4f}")
print(f"Improvement over LR         : {f1_score(y_test, final_model.predict(X_test)) - lr_f1:+.4f}")

# ── Save CV results ───────────────────────────────────────────
cv_results = pd.DataFrame(search.cv_results_)
cv_results.to_csv(f"{PROJECT_ROOT}/outputs/stage8_cv_results.csv", index=False)

# ── Save models to Drive ──────────────────────────────────────
joblib.dump(final_model, f"{PROJECT_ROOT}/outputs/svm_model.joblib")
joblib.dump(lr,          f"{PROJECT_ROOT}/outputs/lr_baseline.joblib")
joblib.dump(search,      f"{PROJECT_ROOT}/outputs/grid_search_results.joblib")

print(f"\n✅ Stage 8 complete.")
print(f"   Primary model : {model_note}")
print(f"   Saved to      : {PROJECT_ROOT}/outputs/svm_model.joblib")

Loading features from Drive...
Training on 20014 segments, 1030 features
Test set: 2312 segments
Train label distribution — Native: 8248, Non-Native: 11766

Running GridSearchCV...
(This may take 10–40 min on Colab CPU — leave the tab open)
Fitting 5 folds for each of 16 candidates, totalling 80 fits


In [ ]:
# ============================================================
# STAGE 9: EVALUATION
# Metrics: Accuracy, F1 (primary), Precision, Recall,
#          ROC AUC, EER, Confusion Matrix, 95% CI
#
# COLAB NOTES:
#   - Pure CPU — no GPU needed, runs in seconds
#   - Results saved to Drive as stage9_metrics.csv
# ============================================================
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import (f1_score, confusion_matrix, accuracy_score,
                              roc_curve, precision_score, recall_score,
                              roc_auc_score)
from scipy import stats

# ── Google Drive path config ─────────────────────────────────────
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

os.makedirs(f"{PROJECT_ROOT}/outputs", exist_ok=True)

# ── Load data and model ───────────────────────────────────────────
print("Loading model and test features...")
data       = np.load(f"{PROJECT_ROOT}/outputs/stage7_features.npz")
X_test     = data["X_test"]
y_test     = data["y_test"]
best_model = joblib.load(f"{PROJECT_ROOT}/outputs/svm_model.joblib")

y_pred   = best_model.predict(X_test)
y_scores = best_model.predict_proba(X_test)[:, 1]   # P(Native)

# ── Metrics ───────────────────────────────────────────────────────
acc       = accuracy_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall    = recall_score(y_test, y_pred, zero_division=0)
auc       = roc_auc_score(y_test, y_scores)
cm        = confusion_matrix(y_test, y_pred)

# EER: threshold where FPR == FRR
fpr, tpr, thresholds = roc_curve(y_test, y_scores)
frr     = 1 - tpr
eer_idx = np.argmin(np.abs(fpr - frr))
eer     = (fpr[eer_idx] + frr[eer_idx]) / 2
eer_thr = thresholds[eer_idx]

# 95% Binomial CI on accuracy
n = len(y_test)
ci_low, ci_high = stats.binom.interval(0.95, n, acc)
ci_low  /= n
ci_high /= n

# ── Print report ──────────────────────────────────────────────────
sep = "=" * 55
print(f"\n{sep}")
print("  EVALUATION REPORT — Stage 9")
print(sep)
print(f"\n  Test set size : {n} segments")
print(f"\n  Accuracy  : {acc*100:.2f}%")
print(f"  F1 Score  : {f1:.4f}  ← primary metric")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  ROC AUC   : {auc:.4f}")
print(f"  EER       : {eer*100:.2f}%  (threshold={eer_thr:.4f})")
print(f"\n  95% CI on Accuracy: [{ci_low*100:.1f}%, {ci_high*100:.1f}%]")
print(f"\n  Confusion Matrix:")
print(f"  (Rows = True class, Cols = Predicted class)")
print(f"              Pred Non-Native  Pred Native")
print(f"  True Non-Nat   {cm[0,0]:>6}          {cm[0,1]:>6}")
print(f"  True Native    {cm[1,0]:>6}          {cm[1,1]:>6}")
print(f"\n  False Positives (non-native → native) : {cm[0,1]}")
print(f"  False Negatives (native → non-native) : {cm[1,0]}")
print(sep)

if eer < 0.05:
    print("  EER < 5%   → Excellent, production-grade")
elif eer < 0.10:
    print("  EER 5–10%  → Good, solid research-grade")
elif eer < 0.20:
    print("  EER 10–20% → Acceptable")
else:
    print("  EER > 20%  → Needs improvement")
print(sep)

# ── Dialect breakdown ─────────────────────────────────────────────
try:
    seg_df   = pd.read_csv(f"{PROJECT_ROOT}/outputs/stage5_manifest.csv")
    test_idx = data["test_idx"]
    test_segs = seg_df.iloc[test_idx].copy()
    test_segs["y_pred"]  = y_pred
    test_segs["y_true"]  = y_test
    test_segs["correct"] = (y_pred == y_test)

    print("\n  Accuracy by dialect:")
    dialect_acc = test_segs.groupby("dialect")["correct"].mean()
    for dialect, acc_d in dialect_acc.items():
        n_d = test_segs[test_segs["dialect"] == dialect].shape[0]
        print(f"    {dialect:<20} {acc_d*100:.1f}%  (n={n_d})")
except Exception as e:
    print(f"\n  (Could not compute dialect breakdown: {e})")

# ── Save metrics ──────────────────────────────────────────────────
metrics = {
    "accuracy":      acc,
    "f1_score":      f1,
    "precision":     precision,
    "recall":        recall,
    "roc_auc":       auc,
    "eer":           eer,
    "eer_threshold": eer_thr,
    "ci_low":        ci_low,
    "ci_high":       ci_high,
    "n_test":        n,
    "tn": cm[0,0], "fp": cm[0,1],
    "fn": cm[1,0], "tp": cm[1,1],
}
pd.DataFrame([metrics]).to_csv(f"{PROJECT_ROOT}/outputs/stage9_metrics.csv", index=False)
print(f"\nSaved: {PROJECT_ROOT}/outputs/stage9_metrics.csv")

In [ ]:
# ============================================================
# STAGE 10: OUTPUT GENERATION
# Deliverables:
#   1. outputs/predictions.csv
#   2. outputs/technical_report.md
#
# COLAB NOTES:
#   - Runs in seconds — pure CPU, no heavy computation
#   - Both output files are saved directly to Drive
# ============================================================
import os
import numpy as np
import pandas as pd
import joblib
from datetime import date

# ── Google Drive path config ─────────────────────────────────────
PROJECT_ROOT = "/content/drive/MyDrive/team_databaes"

os.makedirs(f"{PROJECT_ROOT}/outputs", exist_ok=True)

# ── Load everything ───────────────────────────────────────────────
print("Loading model, features, and metrics...")
data       = np.load(f"{PROJECT_ROOT}/outputs/stage7_features.npz")
X_test     = data["X_test"]
y_test     = data["y_test"]
test_idx   = data["test_idx"]
best_model = joblib.load(f"{PROJECT_ROOT}/outputs/svm_model.joblib")
metrics    = pd.read_csv(f"{PROJECT_ROOT}/outputs/stage9_metrics.csv").iloc[0]

# ── Predictions ───────────────────────────────────────────────────
y_pred     = best_model.predict(X_test)
probs      = best_model.predict_proba(X_test)
confidence      = probs.max(axis=1)
prob_native     = probs[:, 1]
prob_non_native = probs[:, 0]

# ── Load segment info ─────────────────────────────────────────────
try:
    seg_df    = pd.read_csv(f"{PROJECT_ROOT}/outputs/stage5_manifest.csv")
    test_segs = seg_df.iloc[test_idx].reset_index(drop=True)
    sample_ids  = test_segs["seg_id"].tolist()
    parent_ids  = test_segs["parent_id"].tolist()
    dialects    = test_segs["dialect"].tolist()
except Exception as e:
    print(f"Warning: could not load segment info ({e}). Using generic IDs.")
    sample_ids = [f"test_{i:03d}" for i in range(len(y_test))]
    parent_ids = ["unknown"] * len(y_test)
    dialects   = ["unknown"] * len(y_test)

# ── predictions.csv ───────────────────────────────────────────────
results = pd.DataFrame({
    "sample_id":        sample_ids,
    "parent_recording": parent_ids,
    "dialect":          dialects,
    "predicted_label":  y_pred,
    "predicted_class":  ["Native" if p == 1 else "Non-Native" for p in y_pred],
    "confidence_score": np.round(confidence, 4),
    "prob_native":      np.round(prob_native, 4),
    "prob_non_native":  np.round(prob_non_native, 4),
    "true_label":       y_test,
    "true_class":       ["Native" if t == 1 else "Non-Native" for t in y_test],
    "correct":          (y_pred == y_test),
})

predictions_path = f"{PROJECT_ROOT}/outputs/predictions.csv"
results.to_csv(predictions_path, index=False)
n_correct = results["correct"].sum()
n_total   = len(results)
print(f"predictions.csv saved ({n_total} rows, {n_correct}/{n_total} correct)")

# ── Load best params for report ───────────────────────────────────
try:
    grid_search     = joblib.load(f"{PROJECT_ROOT}/outputs/grid_search_results.joblib")
    best_params     = grid_search.best_params_
    best_cv_f1      = grid_search.best_score_
    best_params_str = f"C={best_params.get('C','N/A')}, gamma={best_params.get('gamma','N/A')}"
except Exception:
    best_params_str = "See stage8 logs"
    best_cv_f1      = float(metrics.get("f1_score", 0))

# ── technical_report.md ───────────────────────────────────────────
n_train = int(data["X_train"].shape[0])
n_test  = int(len(y_test))
today   = date.today().strftime("%B %d, %Y")

report = f"""# Technical Evaluation Report
## Team Databaes — Arabic Native/Non-Native Speech Classification
**Client:** Renan Partners Private Limited
**Date:** {today}

---

## 1. Executive Summary

A binary classifier was developed to distinguish Native from Non-Native Arabic speakers.
The model combines deep phonetic representations from a pretrained wav2vec2 model with
hand-crafted prosodic features, trained on Renan recordings and Mozilla Common Voice Arabic data.

**Final Results on Renan Test Set (n={n_test} segments):**

| Metric | Value |
|--------|-------|
| Accuracy | {metrics['accuracy']*100:.2f}% |
| F1 Score | {metrics['f1_score']:.4f} |
| Precision | {metrics['precision']:.4f} |
| Recall | {metrics['recall']:.4f} |
| ROC AUC | {metrics['roc_auc']:.4f} |
| EER | {metrics['eer']*100:.2f}% |
| 95% CI (Accuracy) | [{metrics['ci_low']*100:.1f}%, {metrics['ci_high']*100:.1f}%] |

---

## 2. Methodology

### 2.1 Data
- **Training source:** 160 Renan recordings (Gulf dialects: SA/QA/AE/KW) +
  200 Mozilla Common Voice 24.0 Arabic clips (native reference)
- **Augmentation:** Time stretching (±10%), pitch shifting (±2 semitones), Gaussian noise —
  training segments only. Non-native (minority) received 5 augmentations; native received 1.

### 2.2 Feature Extraction

**Stream 1 — Phonetic Embeddings (768-dim)**
- Model: `jonatasgrosman/wav2vec2-large-xlsr-53-arabic` (Baevski et al., 2020)
- Used frozen — no fine-tuning (prevents overfitting on 160 training recordings)
- Each 3-second segment → mean-pooled hidden states → 768-dimensional vector

**Stream 2 — Prosodic Features (6-dim)**

| Feature | Description |
|---------|-------------|
| F0 mean | Average fundamental frequency |
| F0 variance | Pitch dynamics |
| Speaking rate | Syllable-proxy peaks per second |
| Pause frequency | Pauses (>200ms) per second |
| Mean pause duration | Average pause length in seconds |
| nPVI | Normalised Pairwise Variability Index (rhythmic stress-timing) |

### 2.3 Fusion and Normalisation
- Streams concatenated → 774-dimensional vector per segment
- StandardScaler fitted on training data only

### 2.4 Classifier
- **Algorithm:** SVM, RBF kernel
- **Best hyperparameters:** {best_params_str}
- **Best CV F1:** {best_cv_f1:.4f}
- `class_weight='balanced'`, `probability=True` (Platt scaling for confidence scores)

---

## 3. Results

### 3.1 Confusion Matrix

|  | Predicted Non-Native | Predicted Native |
|--|--|--|
| **True Non-Native** | {int(metrics['tn'])} | {int(metrics['fp'])} |
| **True Native** | {int(metrics['fn'])} | {int(metrics['tp'])} |

- **False Positives** (non-native classified as native): {int(metrics['fp'])}
- **False Negatives** (native classified as non-native): {int(metrics['fn'])}

### 3.2 Statistical Caveat
The 95% CI [{metrics['ci_low']*100:.1f}%, {metrics['ci_high']*100:.1f}%] is relatively wide due to the
small test set (n={n_test}). Results would be more stable with n≥100 test samples.

---

## 4. Known Limitations

1. **Small test set** — wide confidence intervals
2. **Kuwaiti dialect** — only ~1.9% of training data (3 files); may generalise poorly to KW test samples
3. **Short Common Voice clips** — 3–10s clips yield less reliable prosodic estimates
4. **Non-native L1 diversity** — model trained primarily on Gulf-native speakers

---

## 5. References

- Baevski et al. (2020). wav2vec 2.0. https://arxiv.org/abs/2006.11477
- Emara & Shaker (2024). Speech Communication, 157.
- Grabe & Low (2002). Durational variability and the Rhythm Class Hypothesis.
- Sharma et al. (2021). SVM-based speech classification.
- Mozilla Common Voice 24.0 Arabic, CC0-1.0.
- jonatasgrosman/wav2vec2-large-xlsr-53-arabic. https://huggingface.co/jonatasgrosman/wav2vec2-large-xlsr-53-arabic

---
*Report generated automatically by Stage 10 — Team Databaes*
"""

report_path = f"{PROJECT_ROOT}/outputs/technical_report.md"
with open(report_path, "w") as f:
    f.write(report)

print("technical_report.md saved")
print("\n" + "="*50)
print("  ALL STAGE 10 DELIVERABLES COMPLETE")
print("="*50)
print(f"  {predictions_path}")
print(f"  {report_path}")
print("="*50)